In [38]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

In [39]:
base_url = "https://imsdb.com/"
start_url = "https://imsdb.com/all-scripts.html"

In [40]:
# gets the page with all script links
response = requests.get(start_url)
soup = BeautifulSoup(response.text, "html.parser")

# find links to movie pages
movie_links = []

for a in soup.select("p a"):
    href = a.get("href")
    if href and href.startswith("/Movie Scripts/"):
        movie_links.append(urljoin(base_url, href))

print("Movie pages found:", len(movie_links))
print(movie_links[:10])


Movie pages found: 1297
['https://imsdb.com/Movie Scripts/10 Things I Hate About You Script.html', 'https://imsdb.com/Movie Scripts/12 Script.html', 'https://imsdb.com/Movie Scripts/12 and Holding Script.html', 'https://imsdb.com/Movie Scripts/12 Monkeys Script.html', 'https://imsdb.com/Movie Scripts/12 Years a Slave Script.html', 'https://imsdb.com/Movie Scripts/127 Hours Script.html', 'https://imsdb.com/Movie Scripts/1492: Conquest of Paradise Script.html', 'https://imsdb.com/Movie Scripts/15 Minutes Script.html', 'https://imsdb.com/Movie Scripts/17 Again Script.html', 'https://imsdb.com/Movie Scripts/187 Script.html']


In [41]:
def extract_comments(movie_soup): # comments
    comments = []

    text_lines = movie_soup.get_text("\n", strip=True).split("\n")

    capture = False
    current_comment = []
    for line in text_lines:
        if "User Comments for" in line: #start
            capture = True
            continue
        if "Add your own comment" in line: #end
            break
        if capture:
            if line == "* * *": # divider of comments
                if current_comment:
                    comments.append(" ".join(current_comment))
                    current_comment = []
            else:
                current_comment.append(line)

    if current_comment:
        comments.append(" ".join(current_comment))

    return comments

In [42]:
data = []

for movie_url in movie_links[:1000]:   # movies

    movie_page = requests.get(movie_url)
    movie_soup = BeautifulSoup(movie_page.text, "html.parser")

    # get title
    title = movie_soup.title.get_text(strip=True).split(" Script")[0]

    # find script link
    script_link = None
    for a in movie_soup.find_all("a"):
        if "Read" in a.get_text():
            script_link = urljoin(base_url, a.get("href"))
            break
    #continues if script does not exist
    if not script_link:
        print(f"No script found for {title}")
        continue

    comments = extract_comments(movie_soup) #for comments

    # open script page
    script_page = requests.get(script_link)
    script_soup = BeautifulSoup(script_page.text, "html.parser")

    script_container = script_soup.find("td", class_="scrtext")

    if script_container:
        script_text = script_container.get_text("\n", strip=True)

        data.append({
            "title": title,
            "script": script_text,
            "comments": comments
        })


df = pd.DataFrame(data)
print(df.head())

No script found for A.I.
No script found for Batman and Robin
No script found for Batman Begins
No script found for Batman Forever
No script found for Batman Returns
No script found for Casablanca
No script found for Contact
No script found for Dark Knight, The
No script found for Donnie Darko
No script found for Executive Decision
No script found for Eyes Wide Shut
No script found for Full Metal Jacket
No script found for Fury
No script found for Ginger Snaps
No script found for Goodfellas
No script found for Harry Potter and the Chamber of Secrets
No script found for Harry Potter and the Deathly Hallows Part 1
No script found for Harry Potter and the Goblet of Fire
No script found for Harry Potter and the Half-Blood Prince
No script found for Harry Potter and the Prisoner of Azkaban
No script found for Harry Potter and the Sorcerer's Stone
No script found for Innerspace
No script found for Lethal Weapon
No script found for Lethal Weapon 4
No script found for Matchstick Men
No script 

In [43]:
import re
import numpy as np
import pandas as pd


MANUAL_GENRE_VECTORS = {
    "Horror": [
        "blood", "scream", "killer", "dark", "death", "fear", "ghost", "knife", "night", "evil",
        "dead", "body", "monster", "shadow", "haunted", "attack", "hide", "danger", "panic", "cry",
        "terrified", "grave", "curse", "demon", "victim", "murder", "skull", "forest", "shock", "creature"
    ],
    "Crime": [
        "police", "money", "gun", "car", "bank", "street", "boss", "deal", "stole", "detective",
        "robbery", "murder", "case", "criminal", "evidence", "cop", "mafia", "hit", "jail", "drug",
        "suspect", "thief", "lawyer", "court", "gang", "escape", "witness", "track", "investigate", "bullet"
    ],
    "War": [
        "army", "battle", "soldier", "enemy", "captain", "general", "attack", "bomb", "fight", "weapon",
        "mission", "troop", "tank", "warfare", "gunfire", "uniform", "commander", "base", "front", "march",
        "victory", "defeat", "radio", "orders", "camp", "battlefield", "explosion", "rifle", "navy", "aircraft"
    ],
    "Drama": ["emotion", "emotional", "crying", "fight", "fighting", "betrayal", "argument", "regret", "love",
        "cheat", "cheating", "pain", "loss", "grief", "nostalgia", "frustrated", "confused", "miscalculations", 
        "misunderstanding", "significant", "partner", "decision", "choice", "regret", "memory", "memories", "sad",
        "life", "change", "changes", "trust", "lies", "lying", "conflict", "bond", "bonded"
    ],
    "Thriller": ["murder", "crime", "secret", "suspense", "dangerous", "killer", "victim", "traitor", "spy", "spies",
        "agent", "weapon", "weapons", "gun", "guns", "investigation", "hidden", "suspense", "tense", "trap", 
        "escape", "night", "dark", "revenge", "observe", "observed", "evidence", "clue", "betrayal", "panic",
        "discover", "hide", "hiding", "sudden", "suddenly"
    ],
    "Fantasy": ["magic", "wizard", "witch", "spell", "alchemy", "amulet", "armies", "armor", "arrow", "axe", "barbarian",
        "battle", "beast", "bewitch","bones", "brave", "castle", "cauldron", "cave", "centaur", "chamber", "chariot",
        "charm", "conjure", "crown", "curse", "devil", "dragon","dream", "druid", "dungeon", "dwarves", "eerie", "elf", 
        "enchanted", "enchantress", "faerie", "fairy", "fang", "fantastical", "ghoul", "giant", "glitter", "goblin", "hermit", 
        "heroic", "horns"
    ],
    "Musical": ["accompaniment", "actor", "actress", "aria", "arrangement", "audience", "audition", "backstage", "ballad", "band", 
        "beat", "belt", "cabaret","camera", "cast", "celebration", "character", "chorus", "choreographer", "climax", "composer",
        "composition", "conductor", "conductor", "costume", "cresendo", "cue", "curtain", "dance", "dancer", "debut", "director", "diva",
        "duet", "encore", "ensemble", "entertainment", "expression", "finale", "formation", "gesture", "glamor", "groove", 
        "harmony", "improv", "instrument", "interlude", "jazz", "kickline", "lead", "lights", "lyrics", "melody", "microphone", "montage", "musical", 
        "orchestra", "performance", "rhythem", "sing", "singer", "song", "spotlight", "stage", "story", "swing", "tap", "tempo", "tenor", 
        "theater", "tone", "twirl", "vibrato", "voice"
    ],
    "SCI-FI": ["android", "anomaly", "antimatter", "artifact", "artificial", "astronaut", "atmosphere", "avatar", "biomechanical", 
        "biotechnology", "blackhole", "civilization", "clone", "colony", "computer", "conciousness", "cosmic", "cryochamber", "cyberspace", "data", 
        "dimension", "discovery", "drone", "empire", "energy", "evolution", "experiment", "exploration", "extraterrestrial", "forcefield", "future", 
        "galaxy", "generator", "genetic", "gravity", "hologram","hyperspace", "interdimensional", "invasion", "laboratory", "laser", "lifeform", "machine", 
        "meteor", "microchip", "mutation", "neural", "organism", "particle", "photon", "planet", "portal", "quantum", "radiation", "research", "robot", "satellite", 
        "scientist", "spaceship", "supernova", "teleportation", "timewarp", "wormhole"
    ],
    "Action": ["go", "run", "move", "hurry", "stop", "wait", "look", "get", "take", "come", "help", "shoot", "kill", "fight", "attack", "cover", "down", "now", 
        "fast", "quick", "gun", "bomb", "target", "enemy", "mission", "escape", "chase", "danger", "secure", "clear"
    ],
    "Comedy": ["funny", "joke", "laugh", "crazy", "weird", "silly", "oops", "awkward", "sorry", "what", "why", "really", "classic", "wait", "look", 
        "dude", "bro", "man", "cousin", "guy", "exactly", "literally", "party", "drink", "dance","chill", "stupid", "idiot", "seriously", "relax"
    ],
    "Family": ["secret", "believe", "magic", "legend", "discover", "hidden", "forever", "imagine", "journey", "wonder", "together", "promise", "brave", "listen", "heart", 
        "home", "belong", "trust", "change", "remember", "hurry", "escape", "trouble", "monster", "finally", "impossible", "danger", "team", "mission", "destiny"
    ],
    "Romance": ["love", "heart", "kiss", "romance", "romantic", "passion", "darling", "sweet", "desire", "affection", "embrace", "devotion", "soulmate", "beloved", "tender", "date",
        "candlelight", "roses", "charm", "attraction", "intimacy", "admire", "adorable", "flirt", "longing", "connection", "forever", "together", "dream", "hug", "cherish"
    ],
    "Mystery": ["murder", "detective", "clue", "suspect", "alibi", "investigation", "evidence", "crime", "secret", "witness", "disappearance", "case", "killer", "police", 
        "scene", "shadow", "truth", "lie", "victim", "fingerprint", "interrogation", "motive", "puzzle", "hidden", "message", "locked", "reveal", "trail", "danger", "solve"
    ]
    
}

STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from", "had", "has", "have",
    "he", "her", "hers", "him", "his", "i", "if", "in", "into", "is", "it", "its", "me", "my",
    "of", "on", "or", "our", "she", "that", "the", "their", "them", "they", "this", "to", "was",
    "we", "were", "with", "you", "your"
}

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    return [word for word in tokens if word not in STOP_WORDS and len(word) > 1]

def read_text_file(file_name):
    with open(file_name, "r", encoding="utf-8") as f:
        return f.read()

def count_matches(tokens, genre_words):
    counts = pd.Series(tokens).value_counts()
    return sum(counts.get(word, 0) for word in genre_words)

def cosine_similarity(vec1, vec2):
    denom = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    if denom == 0:
        return 0.0
    return float(np.dot(vec1, vec2) / denom)

for movie in data:

    title = movie["title"]
    script_text = movie["script"]
    comments = movie["comments"]

    comments_text = " ".join(comments) if comments else "" #joins script texts and comment texts
    full_text = script_text + " " + comments_text

    tokens = clean_text(full_text)

    vocabulary = []
    for words in MANUAL_GENRE_VECTORS.values():
        for word in words:
            if word not in vocabulary:
                vocabulary.append(word)

    script_vector = np.array([tokens.count(word) for word in vocabulary], dtype=float)

    scores = {}
    for genre, words in MANUAL_GENRE_VECTORS.items():
        genre_vector = np.array([1 if word in words else 0 for word in vocabulary], dtype=float)
        scores[genre] = cosine_similarity(script_vector, genre_vector)

    print("\n-------------------------")
    print("Movie:", title)

    print("Similarity scores:")
    for genre, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
        print(genre, round(score, 4))

    print("\nRaw matching word counts:")
    for genre, words in MANUAL_GENRE_VECTORS.items():
        print(genre, count_matches(tokens, words))

    predicted_genres = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
    print("\nPredicted genres:")
    for genre, score in predicted_genres:
        print(genre)
    break


-------------------------
Movie: 10 Things I Hate About You
Similarity scores:
Comedy 0.3152
Action 0.2888
Romance 0.1087
Horror 0.079
Drama 0.0675
Thriller 0.0638
Crime 0.0598
Family 0.0588
Musical 0.052
War 0.0193
Mystery 0.0172
Fantasy 0.0081
SCI-FI 0.0049

Raw matching word counts:
Horror 78
Crime 59
War 19
Drama 72
Thriller 67
Fantasy 10
Musical 80
SCI-FI 7
Action 285
Comedy 311
Family 58
Romance 109
Mystery 17

Predicted genres:
Comedy
Action
Romance


In [44]:
# testing individual movies
movie_name = "10 Things I Hate About You"

for movie in data:

    title = movie["title"]
    if title != movie_name:
        continue
    script_text = movie["script"]
    comments = movie["comments"]

    comments_text = " ".join(comments) if comments else "" #joins script texts and comment texts
    full_text = script_text + " " + comments_text

    tokens = clean_text(full_text)

    vocabulary = []
    for words in MANUAL_GENRE_VECTORS.values():
        for word in words:
            if word not in vocabulary:
                vocabulary.append(word)

    script_vector = np.array([tokens.count(word) for word in vocabulary], dtype=float)

    scores = {}
    for genre, words in MANUAL_GENRE_VECTORS.items():
        genre_vector = np.array([1 if word in words else 0 for word in vocabulary], dtype=float)
        scores[genre] = cosine_similarity(script_vector, genre_vector)

    print("Movie:", movie["title"])

    print("Similarity scores:")
    for genre, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
        print(genre, round(score, 4))

    print("\nRaw matching word counts:")
    for genre, words in MANUAL_GENRE_VECTORS.items():
        print(genre, count_matches(tokens, words))

    predicted_genres = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
    print("\nPredicted genres:")
    for genre, score in predicted_genres:
        print(genre)
    break

Movie: 10 Things I Hate About You
Similarity scores:
Comedy 0.3152
Action 0.2888
Romance 0.1087
Horror 0.079
Drama 0.0675
Thriller 0.0638
Crime 0.0598
Family 0.0588
Musical 0.052
War 0.0193
Mystery 0.0172
Fantasy 0.0081
SCI-FI 0.0049

Raw matching word counts:
Horror 78
Crime 59
War 19
Drama 72
Thriller 67
Fantasy 10
Musical 80
SCI-FI 7
Action 285
Comedy 311
Family 58
Romance 109
Mystery 17

Predicted genres:
Comedy
Action
Romance
